In [1]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import yaml

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(os.path.join(PROJECT_ROOT, "src"))
from plotting_utils import save_fig
from leakage_audit import (deterministic_sum_check, label_threshold_summary,
                            naive_leaky_cv_accuracy, trivial_threshold_accuracy,
                            sdoh_sniff_test_auc, missingness_dependency_check)

with open(os.path.join(PROJECT_ROOT, "config", "config.yaml")) as f:
    config = yaml.safe_load(f)

SEED = config["seed"]
RAW_DIR = os.path.join(PROJECT_ROOT, config["paths"]["raw_dir"])
TABLES_DIR = os.path.join(PROJECT_ROOT, config["paths"]["tables_dir"])
os.makedirs(TABLES_DIR, exist_ok=True)

In [2]:
pk = pd.read_csv(os.path.join(RAW_DIR, config["raw_files"]["pakistan_raw"]))

phq_cols = [
    "Little interest or pleasure in doing things",
    "Feeling down, depressed, or hopeless",
    "Trouble falling or staying sleep or sleeping too much",
    "Feeling tired or having little energy",
    "Poor appetite or overeating",
    "Feeling badabout yourself that you are failure or have let yourself or your family down",
    "Trouble concentrating on things, such as reading the newspaper or watching television ",
    "Moving or speaking so slowly that other people could have Noticed. ",
    "Thoughts that you would be better off dead, or of hurting yourself",
]

match_rate, mean_abs_diff = deterministic_sum_check(pk, phq_cols, "Scalling")
print(f"Fraction of rows where sum(PHQ-9 items) == Scalling: {match_rate}")
print(f"Mean absolute difference: {mean_abs_diff}")

Fraction of rows where sum(PHQ-9 items) == Scalling: 1.0
Mean absolute difference: 0.0


In [3]:
print(label_threshold_summary(pk, "Scalling", "Labelling"))
# Expect: Depressed range 10-24, Not range 1-9 — confirms Labelling = threshold(Scalling) at cutoff 10

           min  max  count
Labelling                 
Depressed   10   24   9078
Not          1    9   4930


In [4]:
y_pk = (pk["Labelling"] == "Depressed").astype(int)

leaky_acc, leaky_std = naive_leaky_cv_accuracy(pk[phq_cols], y_pk, seed=SEED)
print(f"Naive model (PHQ-9 items -> label), 5-fold CV accuracy: {leaky_acc:.4f} (+/- {leaky_std:.4f})")

trivial_acc = trivial_threshold_accuracy(pk["Scalling"], y_pk, threshold=10)
print(f"Trivial rule (Scalling >= 10), no model at all, accuracy: {trivial_acc:.4f}")

Naive model (PHQ-9 items -> label), 5-fold CV accuracy: 1.0000 (+/- 0.0000)
Trivial rule (Scalling >= 10), no model at all, accuracy: 1.0000


In [5]:
sdoh_cols_pk = ["Age", "Gestational Age", "Number of sons ", "Number of daughters",
                "Total Number of Children", "Gravida", "Female Education", "Husband Education",
                "Working Status", "Physical Health ", "Previous Miscarriage",
                "Sufficient Money for Basic Needs", "Current Appereance Acceptance",
                "Family System", "Male Gender Preference", "Relationship with Mother in-law"]

sdoh_auc, sdoh_std = sdoh_sniff_test_auc(pk[sdoh_cols_pk], y_pk, seed=SEED)
print(f"SDoH-only sniff-test (untuned LogisticRegression), 5-fold CV AUC: {sdoh_auc:.4f} (+/- {sdoh_std:.4f})")
print("\nThis is the realistic, non-circular predictive task. The tuned ensemble in later")
print("notebooks should improve meaningfully on this untuned baseline.")

SDoH-only sniff-test (untuned LogisticRegression), 5-fold CV AUC: 0.5646 (+/- 0.0105)

This is the realistic, non-circular predictive task. The tuned ensemble in later
notebooks should improve meaningfully on this untuned baseline.


In [6]:
comparison_pk = pd.DataFrame({
    "Approach": ["Published PERI_DEP claim (RNN-LSTM+SMOTE/GAN)",
                 "Reproduced leaky model (PHQ items -> label)",
                 "Trivial rule (Scalling >= 10), no model",
                 "Legitimate SDoH-only model (untuned)"],
    "Metric": ["Accuracy", "Accuracy", "Accuracy", "AUC-ROC"],
    "Value": [0.95, round(leaky_acc, 4), round(trivial_acc, 4), round(sdoh_auc, 4)],
    "Uses outcome-defining items as predictors": ["Yes (likely)", "Yes", "N/A (direct rule)", "No"],
})
print(comparison_pk)
comparison_pk.to_csv(os.path.join(TABLES_DIR, "pakistan_leakage_comparison.csv"), index=False)

                                        Approach    Metric   Value  \
0  Published PERI_DEP claim (RNN-LSTM+SMOTE/GAN)  Accuracy  0.9500   
1    Reproduced leaky model (PHQ items -> label)  Accuracy  1.0000   
2        Trivial rule (Scalling >= 10), no model  Accuracy  1.0000   
3           Legitimate SDoH-only model (untuned)   AUC-ROC  0.5646   

  Uses outcome-defining items as predictors  
0                              Yes (likely)  
1                                       Yes  
2                         N/A (direct rule)  
3                                        No  


In [7]:
ar = pd.read_excel(os.path.join(RAW_DIR, config["raw_files"]["arab5_data"]))

dep_items = [f"DepressionRec{i}" for i in range(1, 9)] + ["Depressionec9"]
anx_items = [f"Anxietyrec{i}" for i in range(1, 8)]

match_rate_dep, mad_dep = deterministic_sum_check(ar, dep_items, "PHQ_9_Depression_Score_first")
print(f"Depression items -> score match rate: {match_rate_dep}, mean abs diff: {mad_dep}")

Depression items -> score match rate: 1.0, mean abs diff: 0.0


In [8]:
ar["dep_sum"] = ar[dep_items].sum(axis=1, skipna=False)
ar["anx_sum"] = ar[anx_items].sum(axis=1, skipna=False)

print("Depression category thresholds (standard PHQ-9 cutoffs):")
print(label_threshold_summary(ar.dropna(subset=dep_items), "dep_sum", "Depression_Rec1"))
print()
print("Anxiety category thresholds (standard GAD-7 cutoffs):")
print(label_threshold_summary(ar.dropna(subset=anx_items), "anx_sum", "Anxeity_Rec"))

Depression category thresholds (standard PHQ-9 cutoffs):
                  min   max  count
Depression_Rec1                   
1                 0.0   9.0    815
2                10.0  14.0    391
3                15.0  27.0    293

Anxiety category thresholds (standard GAD-7 cutoffs):
              min   max  count
Anxeity_Rec                   
1             0.0   4.0    260
2             5.0   9.0    512
3            10.0  17.0    174


In [9]:
n_total_dep1 = (ar["Depression_Rec1"] == 1).sum()
n_dep1_missing_items = ((ar["Depression_Rec1"] == 1) & ar[dep_items].isna().any(axis=1)).sum()
print(f"Rows labeled 'Minimal/Mild' depression (category 1): {n_total_dep1}")
print(f"Of those, rows with missing PHQ-9 items (i.e. no real symptom data, defaulted to category 1): {n_dep1_missing_items}")
print(f"-> {100*n_dep1_missing_items/n_total_dep1:.1f}% of 'Minimal/Mild' labels are not measured, they're a default.")

Rows labeled 'Minimal/Mild' depression (category 1): 2846
Of those, rows with missing PHQ-9 items (i.e. no real symptom data, defaulted to category 1): 2031
-> 71.4% of 'Minimal/Mild' labels are not measured, they're a default.


In [10]:
dep_gap = missingness_dependency_check(ar, dep_items, "part1_country")
print("Fraction of rows with missing depression items, by country code:")
print(dep_gap)
print("\nMissingness rate varies substantially by country -> the 'default to Minimal/Mild'")
print("imputation is NOT random. This is a second, subtler leakage/bias path: country")
print("becomes an indirect predictor of the (partly fabricated) label.")

Fraction of rows with missing depression items, by country code:
col_0          False  True 
part1_country              
1              0.544  0.456
2              0.394  0.606
4              0.394  0.606
6              0.321  0.679
8              0.478  0.522

Missingness rate varies substantially by country -> the 'default to Minimal/Mild'
imputation is NOT random. This is a second, subtler leakage/bias path: country
becomes an indirect predictor of the (partly fabricated) label.


In [11]:
sub_dep = ar.dropna(subset=dep_items).copy()
y_dep = (sub_dep["Depression_Rec1"] >= 2).astype(int)  # Moderate+Severe vs Minimal/Mild

leaky_acc_ar, leaky_std_ar = naive_leaky_cv_accuracy(sub_dep[dep_items], y_dep, seed=SEED)
print(f"Naive model (depression items -> label), 5-fold CV accuracy: {leaky_acc_ar:.4f} (+/- {leaky_std_ar:.4f})")

trivial_acc_ar = trivial_threshold_accuracy(sub_dep["dep_sum"], y_dep, threshold=10)
print(f"Trivial rule (sum >= 10), accuracy: {trivial_acc_ar:.4f}")

Naive model (depression items -> label), 5-fold CV accuracy: 1.0000 (+/- 0.0000)
Trivial rule (sum >= 10), accuracy: 1.0000


In [12]:
sdoh_cols_ar = ["part1_country", "Locality_first", "WomenAge", "MariageAge", "NumberofPregnancy",
                "NumberofAbortions", "EducationLevel", "Work", "Health_Prob", "StressduringCovid",
                "FamilyProblems", "FinancialProblem", "SocialProblem", "PsychologicalProb",
                "WorkStress", "FamilyIncome1"]

y_dep_full = (ar["Depression_Rec1"] >= 2).astype(int)
sdoh_auc_ar, sdoh_std_ar = sdoh_sniff_test_auc(ar[sdoh_cols_ar], y_dep_full, seed=SEED)
print(f"SDoH-only sniff-test AUC (depression, full n, includes default-imputed labels): {sdoh_auc_ar:.4f}")
print("\nNote: this number is likely inflated by the country<->missingness link found in Cell 10.")
print("The formal SDoH-only model (notebook 05b) will need to either exclude default-imputed")
print("rows or model 'item-missingness' explicitly as a sensitivity check.")

SDoH-only sniff-test AUC (depression, full n, includes default-imputed labels): 0.8831

Note: this number is likely inflated by the country<->missingness link found in Cell 10.
The formal SDoH-only model (notebook 05b) will need to either exclude default-imputed
rows or model 'item-missingness' explicitly as a sensitivity check.


In [13]:
sub_anx = ar.dropna(subset=anx_items).copy()
y_anx = (sub_anx["Anxeity_Rec"] >= 2).astype(int)

leaky_acc_anx, _ = naive_leaky_cv_accuracy(sub_anx[anx_items], y_anx, seed=SEED)
trivial_acc_anx = trivial_threshold_accuracy(sub_anx["anx_sum"], y_anx, threshold=5)
print(f"Naive model (anxiety items -> label) CV accuracy: {leaky_acc_anx:.4f}")
print(f"Trivial rule (sum >= 5) accuracy: {trivial_acc_anx:.4f}")

Naive model (anxiety items -> label) CV accuracy: 0.9968
Trivial rule (sum >= 5) accuracy: 1.0000


In [14]:
comparison_ar = pd.DataFrame({
    "Approach": ["Published Qasrawi et al. claim (Gradient Boosting, depression)",
                 "Published Qasrawi et al. claim (Gradient Boosting, anxiety)",
                 "Reproduced leaky model - depression (items -> label)",
                 "Reproduced leaky model - anxiety (items -> label)",
                 "Legitimate SDoH-only model - depression (untuned, caveat: default-imputed labels)"],
    "Metric": ["Accuracy", "Accuracy", "Accuracy", "Accuracy", "AUC-ROC"],
    "Value": [0.833, 0.829, round(leaky_acc_ar, 4), round(leaky_acc_anx, 4), round(sdoh_auc_ar, 4)],
})
print(comparison_ar)
comparison_ar.to_csv(os.path.join(TABLES_DIR, "arab5_leakage_comparison.csv"), index=False)

                                            Approach    Metric   Value
0  Published Qasrawi et al. claim (Gradient Boost...  Accuracy  0.8330
1  Published Qasrawi et al. claim (Gradient Boost...  Accuracy  0.8290
2  Reproduced leaky model - depression (items -> ...  Accuracy  1.0000
3  Reproduced leaky model - anxiety (items -> label)  Accuracy  0.9968
4  Legitimate SDoH-only model - depression (untun...   AUC-ROC  0.8831


In [15]:
ug = pd.read_csv(os.path.join(RAW_DIR, config["raw_files"]["uganda"]))
ug["age_numeric"] = pd.to_numeric(ug["age"], errors="coerce")

numeric_cols_ug = ["age_numeric", "otherChildren", "numStillBorn", "numMissCarry"]
print("Correlation of candidate predictors with Total Score (checking for a hidden deterministic link):")
print(ug[numeric_cols_ug + ["Total Score"]].corr()["Total Score"])
print("\nNo item-level breakdown of Total Score/Anxiety exists in this file, and no candidate")
print("predictor is even moderately correlated with it -> no circular-leakage removal needed")
print("for this dataset. Documented here for audit completeness.")

Correlation of candidate predictors with Total Score (checking for a hidden deterministic link):
age_numeric     -0.070870
otherChildren   -0.044308
numStillBorn    -0.023743
numMissCarry    -0.175175
Total Score      1.000000
Name: Total Score, dtype: float64

No item-level breakdown of Total Score/Anxiety exists in this file, and no candidate
predictor is even moderately correlated with it -> no circular-leakage removal needed
for this dataset. Documented here for audit completeness.


In [16]:
print("post_natal_data.csv has no outcome column at all. The leakage question here is different:")
print("whichever items get used to CONSTRUCT a proxy label in notebook 03 cannot also be used as")
print("predictors of that label. This will be handled explicitly in 03_label_construction.ipynb")
print("by splitting the 9 items into a small 'label-construction subset' and a disjoint")
print("'predictor subset' -- documented the same way as the exclusions above.")

post_natal_data.csv has no outcome column at all. The leakage question here is different:
whichever items get used to CONSTRUCT a proxy label in notebook 03 cannot also be used as
predictors of that label. This will be handled explicitly in 03_label_construction.ipynb
by splitting the 9 items into a small 'label-construction subset' and a disjoint
'predictor subset' -- documented the same way as the exclusions above.


In [17]:
exclusions = pd.DataFrame({
    "Dataset": ["Pakistan", "Pakistan", "Arab5", "Arab5", "Arab5", "Arab5"],
    "Excluded variable(s)": ["9 PHQ-9 item columns", "Scalling",
                              "9 PHQ-9 item columns (DepressionRec1-9)", "PHQ_9_Depression_Score_first",
                              "7 GAD-7 item columns (Anxietyrec1-7)", "GAD_7_Anxiety_score_Cat_first"],
    "Reason": ["Deterministic components of Labelling (match rate 1.0)",
               "Deterministic threshold basis of Labelling",
               "Deterministic components of Depression_Rec1/DepressionCat_first",
               "Deterministic sum basis of Depression_Rec1",
               "Deterministic components of Anxeity_Rec/AnxietyCat_first",
               "Deterministic sum basis of Anxeity_Rec"],
})
print(exclusions)
exclusions.to_csv(os.path.join(TABLES_DIR, "leakage_audit_exclusions.csv"), index=False)
print(f"\nSaved to {os.path.join(TABLES_DIR, 'leakage_audit_exclusions.csv')}")

    Dataset                     Excluded variable(s)  \
0  Pakistan                     9 PHQ-9 item columns   
1  Pakistan                                 Scalling   
2     Arab5  9 PHQ-9 item columns (DepressionRec1-9)   
3     Arab5             PHQ_9_Depression_Score_first   
4     Arab5     7 GAD-7 item columns (Anxietyrec1-7)   
5     Arab5            GAD_7_Anxiety_score_Cat_first   

                                              Reason  
0  Deterministic components of Labelling (match r...  
1         Deterministic threshold basis of Labelling  
2  Deterministic components of Depression_Rec1/De...  
3         Deterministic sum basis of Depression_Rec1  
4  Deterministic components of Anxeity_Rec/Anxiet...  
5             Deterministic sum basis of Anxeity_Rec  

Saved to C:\maternal-mental-health-risk\reports/tables\leakage_audit_exclusions.csv
